# Advanced Semantic Similarity Search on Netflix Titles

This notebook implements a production-style semantic search workflow using Sentence-BERT embeddings and cosine similarity.

What is included in this advanced version:
- clean and reproducible data preparation
- embedding cache to avoid recomputation
- robust retrieval functions with validation and score threshold
- flexible title matching (exact + approximate)
- lightweight evaluation metrics and CSV export

## Step 1 - Setup, Load Data, and Build Features

Import dependencies, define configuration paths, load the dataset, and build a semantic text field used for embedding generation.

In [12]:
import os
import re
from pathlib import Path
from difflib import get_close_matches

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# STEP 1: Setup, Load Data, and Build Features
DATA_PATH = Path("dataset/netflix_titles.csv")
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_PATH = CACHE_DIR / "netflix_all_minilm_embeddings.npy"

print("Loading dataset...")
df = pd.read_csv(DATA_PATH)

required_cols = ["title", "listed_in", "description", "type", "release_year", "rating"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Dataset is missing required columns: {missing_required}")

# Fill NA values for stable preprocessing
df["title"] = df["title"].fillna("Unknown")
df["listed_in"] = df["listed_in"].fillna("")
df["description"] = df["description"].fillna("")
df["type"] = df["type"].fillna("Unknown")
df["rating"] = df["rating"].fillna("Unknown")

# Build a richer semantic feature string
df["combined_features"] = (
    "Title: " + df["title"] + ". " +
    "Genre: " + df["listed_in"] + ". " +
    "Description: " + df["description"]
)

print(f"Dataset shape: {df.shape}")
df[["title", "listed_in", "combined_features"]].head(3)

Loading dataset...
Dataset shape: (8807, 13)


,title,listed_in,combined_features
0,Dick Johnson Is Dead,Documentaries,Title: Dick Johnson Is Dead. Genre: Documentar...
1,Blood & Water,"International TV Shows, TV Dramas, TV Mysteries",Title: Blood & Water. Genre: International TV ...
2,Ganglands,"Crime TV Shows, International TV Shows, TV Act...","Title: Ganglands. Genre: Crime TV Shows, Inter..."


## Step 2 - Initialize Embedding Model

Load a Sentence-BERT model that converts each title record and user query into dense vectors.

In [8]:
# STEP 2: Initialize Embedding Model
MODEL_NAME = "all-MiniLM-L6-v2"
print(f"Loading Sentence-BERT model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)

Loading Sentence-BERT model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Step 3 - Generate or Load Embeddings (Cache Enabled)

Load embeddings from disk when available; otherwise encode all records and save them for faster future runs.

In [9]:
# STEP 3: Generate or Load Embeddings (Cache Enabled)
if EMBEDDING_PATH.exists():
    print(f"Loading embeddings from cache: {EMBEDDING_PATH}")
    document_embeddings = np.load(EMBEDDING_PATH)
else:
    print("Encoding records and creating embedding cache...")
    document_embeddings = model.encode(
        df["combined_features"].tolist(),
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    np.save(EMBEDDING_PATH, document_embeddings)
    print(f"Embeddings saved to: {EMBEDDING_PATH}")

print(f"Embeddings shape: {document_embeddings.shape}")

Encoding records and creating embedding cache...


Batches:   0%|          | 0/276 [00:00<?, ?it/s]

Embeddings saved to: cache\netflix_all_minilm_embeddings.npy
Embeddings shape: (8807, 384)


## Step 4 - Define Retrieval and Evaluation Utilities

Create robust functions for semantic retrieval, title-based retrieval, overlap analysis, and benchmark-style evaluation.

In [13]:
# STEP 4: Define Retrieval and Evaluation Utilities
title_lookup = df["title"].fillna("").astype(str).tolist()
title_lookup_lower = [t.lower() for t in title_lookup]

def _tokenize(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))

def resolve_title(query, cutoff=0.60):
    key = query.strip().lower()
    if not key:
        return None, "empty"

    if key in title_lookup_lower:
        return key, "exact"

    query_tokens = _tokenize(key)
    best_title = None
    best_score = 0.0

    # Token-overlap strategy improves typo handling better than plain fuzzy matching.
    for candidate in title_lookup_lower:
        cand_tokens = _tokenize(candidate)
        if not cand_tokens:
            continue
        jaccard = len(query_tokens & cand_tokens) / len(query_tokens | cand_tokens) if query_tokens else 0.0
        if jaccard > best_score:
            best_score = jaccard
            best_title = candidate

    if best_title is not None and best_score >= 0.50:
        return best_title, "token_overlap"

    candidates = get_close_matches(key, title_lookup_lower, n=1, cutoff=cutoff)
    if candidates:
        return candidates[0], "approx"
    return None, "not_found"

def semantic_search(
    query,
    df,
    embeddings,
    top_n=5,
    min_score=0.0,
    return_columns=None,
):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query must be a non-empty string")
    if top_n <= 0:
        raise ValueError("top_n must be greater than 0")

    if return_columns is None:
        return_columns = ["title", "type", "release_year", "rating", "listed_in", "description"]

    query_vector = model.encode([query], convert_to_numpy=True)
    similarity_scores = cosine_similarity(query_vector, embeddings)[0]
    ranked_indices = similarity_scores.argsort()[::-1]

    rows = []
    for idx in ranked_indices:
        score = float(similarity_scores[idx])
        if score < min_score:
            continue

        row = {
            "rank": len(rows) + 1,
            "query": query,
            "similarity_score": round(score, 4),
        }

        for col in return_columns:
            row[col] = df.iloc[idx][col] if col in df.columns else None

        if "description" in row and isinstance(row["description"], str):
            row["description_preview"] = row["description"][:120] + ("..." if len(row["description"]) > 120 else "")

        rows.append(row)
        if len(rows) >= top_n:
            break

    return pd.DataFrame(rows)

def similar_by_title(title_query, df, embeddings, top_n=5, min_score=0.0, cutoff=0.60):
    resolved_key, match_type = resolve_title(title_query, cutoff=cutoff)
    if resolved_key is None:
        return pd.DataFrame({
            "query_input": [title_query],
            "match_type": [match_type],
            "message": ["Title not found. Try another query."],
        })

    original_title = title_lookup[title_lookup_lower.index(resolved_key)]
    result = semantic_search(
        query=original_title,
        df=df,
        embeddings=embeddings,
        top_n=top_n + 1,
        min_score=min_score,
    )

    if not result.empty and "title" in result.columns:
        result = result[result["title"].str.lower() != resolved_key].head(top_n).copy()

    if result.empty:
        return pd.DataFrame({
            "query_input": [title_query],
            "resolved_title": [original_title],
            "match_type": [match_type],
            "message": ["No recommendations found above the score threshold."],
        })

    result.insert(0, "query_input", title_query)
    result.insert(1, "resolved_title", original_title)
    result.insert(2, "match_type", match_type)
    return result.reset_index(drop=True)

def evaluate_semantic_queries(queries, df, embeddings, top_n=5, min_score=0.0):
    eval_rows = []
    results_by_query = {}

    for query in queries:
        rec = semantic_search(query, df, embeddings, top_n=top_n, min_score=min_score)
        results_by_query[query] = rec

        if rec.empty:
            eval_rows.append({
                "query": query,
                "returned_items": 0,
                "avg_score": np.nan,
                "max_score": np.nan,
                "genre_diversity": 0,
            })
            continue

        genres = rec["listed_in"].fillna("").astype(str).str.split(",")
        genre_tokens = {g.strip().lower() for items in genres for g in items if g.strip()}

        eval_rows.append({
            "query": query,
            "returned_items": len(rec),
            "avg_score": round(float(rec["similarity_score"].mean()), 4),
            "max_score": round(float(rec["similarity_score"].max()), 4),
            "genre_diversity": len(genre_tokens),
        })

    return pd.DataFrame(eval_rows), results_by_query

def pairwise_overlap(results_by_query):
    queries = list(results_by_query.keys())
    overlap_rows = []
    for i in range(len(queries)):
        for j in range(i + 1, len(queries)):
            q1, q2 = queries[i], queries[j]
            s1 = set(results_by_query[q1].get("title", pd.Series(dtype=str)).astype(str))
            s2 = set(results_by_query[q2].get("title", pd.Series(dtype=str)).astype(str))

            denom = max(1, min(len(s1), len(s2)))
            ratio = len(s1.intersection(s2)) / denom
            overlap_rows.append({
                "query_a": q1,
                "query_b": q2,
                "overlap_count": len(s1.intersection(s2)),
                "overlap_ratio": round(ratio, 4),
            })

    return pd.DataFrame(overlap_rows)

## Step 5 - Run Benchmarks and Export Outputs

Run sample semantic and title-based queries, compute summary metrics, and export results for reporting.

In [14]:
# STEP 5: Run Benchmarks and Export Outputs
semantic_queries = [
    "dark magic and fantasy world",
    "investigating a serial killer",
    "funny stand up comedy",
    "slow emotional romance",
    "political crime thriller"
]

title_queries = ["Narcos", "The Crowm", "Money Heist"]

# 1) Run semantic-query retrieval
all_semantic_results = []
for q in semantic_queries:
    rec = semantic_search(
        query=q,
        df=df,
        embeddings=document_embeddings,
        top_n=5,
        min_score=0.20,
    )
    all_semantic_results.append(rec)
    print("=" * 80)
    print(f"Semantic query: {q}")
    display(rec[["rank", "title", "similarity_score", "listed_in", "description_preview"]].head(5))

semantic_results_df = pd.concat(all_semantic_results, ignore_index=True) if all_semantic_results else pd.DataFrame()

# 2) Run flexible title-based retrieval
all_title_results = []
for tq in title_queries:
    t_rec = similar_by_title(
        title_query=tq,
        df=df,
        embeddings=document_embeddings,
        top_n=5,
        min_score=0.20,
        cutoff=0.55,
    )
    all_title_results.append(t_rec)
    print("=" * 80)
    print(f"Title query: {tq}")
    display(t_rec.head(5))

title_results_df = pd.concat(all_title_results, ignore_index=True) if all_title_results else pd.DataFrame()

# 3) Evaluate semantic-query quality metrics
eval_df, results_map = evaluate_semantic_queries(
    queries=semantic_queries,
    df=df,
    embeddings=document_embeddings,
    top_n=5,
    min_score=0.20,
 )
overlap_df = pairwise_overlap(results_map)

print("\nEvaluation summary:")
display(eval_df)
print("\nPairwise overlap between query result sets:")
display(overlap_df)

# 4) Export outputs
semantic_results_path = "semantic_query_results.csv"
title_results_path = "title_query_results.csv"
eval_results_path = "semantic_evaluation_summary.csv"
overlap_results_path = "semantic_pairwise_overlap.csv"

if not semantic_results_df.empty:
    semantic_results_df.to_csv(semantic_results_path, index=False)
if not title_results_df.empty:
    title_results_df.to_csv(title_results_path, index=False)
if not eval_df.empty:
    eval_df.to_csv(eval_results_path, index=False)
if not overlap_df.empty:
    overlap_df.to_csv(overlap_results_path, index=False)

print("\nSaved files:")
print(f"- {semantic_results_path}")
print(f"- {title_results_path}")
print(f"- {eval_results_path}")
print(f"- {overlap_results_path}")

Semantic query: dark magic and fantasy world


,rank,title,similarity_score,listed_in,description_preview
0,1,Shadow,0.5170,"Action & Adventure, Dramas, International Movies",As three kingdoms struggle for control of a wa...
1,2,Light in the Dark,0.5047,"Dramas, International Movies",A terrifying home invasion shatters a couple's...
2,3,The Magicians,0.5020,"TV Dramas, TV Sci-Fi & Fantasy",When grad student Quentin Coldwater enters a c...
3,4,Nightbooks,0.4969,Children & Family Movies,Scary story fan Alex must tell a spine-tinglin...
4,5,The Crystal Calls Making the Dark Crystal: Age...,0.4889,"Documentaries, International Movies","Go behind the scenes with stars, puppeteers an..."


Semantic query: investigating a serial killer


,rank,title,similarity_score,listed_in,description_preview
0,1,Inside the Mind of a Serial Killer,0.5663,"Crime TV Shows, Docuseries",Mixing dramatic re-enactments with real-life f...
1,2,The Investigator: A British Crime Story,0.5466,"British TV Shows, Crime TV Shows, Docuseries","After 40 years of inconclusive evidence, renow..."
2,3,The Murder Detectives,0.5318,"British TV Shows, Crime TV Shows, Docuseries",This series tracks the ups and downs of an 18-...
3,4,I AM A KILLER: RELEASED,0.5271,"British TV Shows, Crime TV Shows, Docuseries","In this crime docuseries spinoff, a convict is..."
4,5,Forensic,0.5212,"International Movies, Thrillers",A pair of officers with history navigates clue...


Semantic query: funny stand up comedy


...


Evaluation summary:


,query,returned_items,avg_score,max_score,genre_diversity
0,dark magic and fantasy world,5,0.5019,0.5170,7
1,investigating a serial killer,5,0.5386,0.5663,5
2,funny stand up comedy,5,0.6159,0.6412,3
3,slow emotional romance,5,0.5486,0.5842,8
4,political crime thriller,5,0.5871,0.5964,7



Pairwise overlap between query result sets:


,query_a,query_b,overlap_count,overlap_ratio
0,dark magic and fantasy world,investigating a serial killer,0,0.0
1,dark magic and fantasy world,funny stand up comedy,0,0.0
2,dark magic and fantasy world,slow emotional romance,0,0.0
3,dark magic and fantasy world,political crime thriller,0,0.0
4,investigating a serial killer,funny stand up comedy,0,0.0
5,investigating a serial killer,slow emotional romance,0,0.0
6,investigating a serial killer,political crime thriller,0,0.0
7,funny stand up comedy,slow emotional romance,0,0.0
8,funny stand up comedy,political crime thriller,0,0.0
9,slow emotional romance,political crime thriller,0,0.0



Saved files:
- semantic_query_results.csv
- title_query_results.csv
- semantic_evaluation_summary.csv
- semantic_pairwise_overlap.csv
